In [1]:
import pandas as pd
import numpy as np

In [2]:
from datasets import load_dataset

# Preparar datos y divisiones de train y test
df = pd.read_csv('DB/youtube.csv')
train_data, test_data = df[df["split"] == "train"], df[df["split"] == "test"]

c:\Program Files\Python\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


FileNotFoundError: [Errno 2] No such file or directory: 'DB/youtube.csv'

In [ ]:
train_data

Dataset({
    features: ['text', 'label'],
    num_rows: 8530
})

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Modelo y tokenizador
model_id = "bert-base-cased"
model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)
tokenizer = AutoTokenizer.from_pretrained(model_id)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

In [ ]:
from transformers import DataCollatorWithPadding

# Rellenar hasta la secuencia más larga.
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def preprocess_function(examples):
   """Tokenize datos de entrada"""
   return tokenizer(examples["text"], truncation=True)

# Tokenize train/test
tokenized_train = train_data.map(preprocess_function, batched=True)
tokenized_test = test_data.map(preprocess_function, batched=True)

Map:   0%|          | 0/8530 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

In [ ]:
import numpy as np
import evaluate

def compute_metrics(eval_pred):
  """Calcular F1 score"""
  logits, labels = eval_pred
  predictions = np.argmax(logits, axis=-1)

  load_f1 = evaluate.load("f1")
  f1 = load_f1.compute(predictions=predictions, references=labels)["f1"]
  return {"f1": f1}

In [ ]:
from transformers import TrainingArguments, Trainer

# Argumentos de entrenamiento para el ajuste de parámetros
training_args = TrainingArguments(
   "model",
   learning_rate=2e-5,
   per_device_train_batch_size=8,
   per_device_eval_batch_size=8,
   num_train_epochs=1,
   weight_decay=0.01,
   save_strategy="epoch",
   report_to="none"
)

# "trainer" ejecuta el proceso de entrenamiento
trainer = Trainer(
   model=model,
   args=training_args,
   train_dataset=tokenized_train,
   eval_dataset=tokenized_test,
   tokenizer=tokenizer,
   data_collator=data_collator,
   compute_metrics=compute_metrics,
)

/tmp/ipython-input-2664990676.py:16: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
trainer.train()

Step,Training Loss
500,0.444900
1000,0.374400


TrainOutput(global_step=1067, training_loss=0.4075525832600759, metrics={'train_runtime': 128.7025, 'train_samples_per_second': 66.277, 'train_steps_per_second': 8.29, 'total_flos': 206462011807920.0, 'train_loss': 0.4075525832600759, 'epoch': 1.0})

In [ ]:
trainer.evaluate()

{'eval_loss': 0.3781662583351135,
 'eval_f1': 0.8576814326107446,
 'eval_runtime': 3.8663,
 'eval_samples_per_second': 275.713,
 'eval_steps_per_second': 34.658,
 'epoch': 1.0}

In [ ]:
trainer.save_model("model2")
tokenizer.save_pretrained("tokenizer2")

('tokenizer2/tokenizer_config.json',
 'tokenizer2/special_tokens_map.json',
 'tokenizer2/vocab.txt',
 'tokenizer2/added_tokens.json',
 'tokenizer2/tokenizer.json')

In [ ]:
loaded_tokenizer = AutoTokenizer.from_pretrained("tokenizer2")
loaded_model = AutoModelForSequenceClassification.from_pretrained("model2")

##Congelar capas

In [ ]:
# Cargar el modelo y el tokenizador
model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)
tokenizer = AutoTokenizer.from_pretrained(model_id)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# Se imprimen los nombres de las capas del modelo
for name, param in model.named_parameters():
  print(name)

bert.embeddings.word_embeddings.weight
bert.embeddings.position_embeddings.weight
bert.embeddings.token_type_embeddings.weight
bert.embeddings.LayerNorm.weight
bert.embeddings.LayerNorm.bias
bert.encoder.layer.0.attention.self.query.weight
bert.encoder.layer.0.attention.self.query.bias
bert.encoder.layer.0.attention.self.key.weight
bert.encoder.layer.0.attention.self.key.bias
bert.encoder.layer.0.attention.self.value.weight
bert.encoder.layer.0.attention.self.value.bias
bert.encoder.layer.0.attention.output.dense.weight
bert.encoder.layer.0.attention.output.dense.bias
bert.encoder.layer.0.attention.output.LayerNorm.weight
bert.encoder.layer.0.attention.output.LayerNorm.bias
bert.encoder.layer.0.intermediate.dense.weight
bert.encoder.layer.0.intermediate.dense.bias
bert.encoder.layer.0.output.dense.weight
bert.encoder.layer.0.output.dense.bias
bert.encoder.layer.0.output.LayerNorm.weight
bert.encoder.layer.0.output.LayerNorm.bias
bert.encoder.layer.1.attention.self.query.weight
bert.enc

In [ ]:
for name, param in model.named_parameters():
  # Cabeza de clasificación entrenable
  if name.startswith("classifier"):
    param.requires_grad = True

  # Freeze everything else
  else:
    param.requires_grad = False

In [ ]:
# Podemos comprobar si el modelo se actualizó correctamente
for name, param in model.named_parameters():
  print(f"Parameter: {name} ----- {param.requires_grad}")

Parameter: bert.embeddings.word_embeddings.weight ----- False
Parameter: bert.embeddings.position_embeddings.weight ----- False
Parameter: bert.embeddings.token_type_embeddings.weight ----- False
Parameter: bert.embeddings.LayerNorm.weight ----- False
Parameter: bert.embeddings.LayerNorm.bias ----- False
Parameter: bert.encoder.layer.0.attention.self.query.weight ----- False
Parameter: bert.encoder.layer.0.attention.self.query.bias ----- False
Parameter: bert.encoder.layer.0.attention.self.key.weight ----- False
Parameter: bert.encoder.layer.0.attention.self.key.bias ----- False
Parameter: bert.encoder.layer.0.attention.self.value.weight ----- False
Parameter: bert.encoder.layer.0.attention.self.value.bias ----- False
Parameter: bert.encoder.layer.0.attention.output.dense.weight ----- False
Parameter: bert.encoder.layer.0.attention.output.dense.bias ----- False
Parameter: bert.encoder.layer.0.attention.output.LayerNorm.weight ----- False
Parameter: bert.encoder.layer.0.attention.output

In [ ]:
from transformers import TrainingArguments, Trainer

# "trainer" ejecuta el proceso de entrenamiento
trainer = Trainer(
   model=model,
   args=training_args,
   train_dataset=tokenized_train,
   eval_dataset=tokenized_test,
   tokenizer=tokenizer,
   data_collator=data_collator,
   compute_metrics=compute_metrics,
)
trainer.train()

/tmp/ipython-input-3611702304.py:4: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
500,0.696900
1000,0.690900


TrainOutput(global_step=1067, training_loss=0.6939712352806313, metrics={'train_runtime': 45.5403, 'train_samples_per_second': 187.307, 'train_steps_per_second': 23.43, 'total_flos': 206462011807920.0, 'train_loss': 0.6939712352806313, 'epoch': 1.0})

In [ ]:
trainer.evaluate()

{'eval_loss': 0.6784813404083252,
 'eval_f1': 0.6055555555555555,
 'eval_runtime': 3.724,
 'eval_samples_per_second': 286.253,
 'eval_steps_per_second': 35.983,
 'epoch': 1.0}

### Congelar capas

In [ ]:
# Podemos comprobar si el modelo se actualizó correctamente
for index, (name, param) in enumerate(model.named_parameters()):
  print(f"Parameter: {index}{name} ----- {param.requires_grad}")

Parameter: 0bert.embeddings.word_embeddings.weight ----- False
Parameter: 1bert.embeddings.position_embeddings.weight ----- False
Parameter: 2bert.embeddings.token_type_embeddings.weight ----- False
Parameter: 3bert.embeddings.LayerNorm.weight ----- False
Parameter: 4bert.embeddings.LayerNorm.bias ----- False
Parameter: 5bert.encoder.layer.0.attention.self.query.weight ----- False
Parameter: 6bert.encoder.layer.0.attention.self.query.bias ----- False
Parameter: 7bert.encoder.layer.0.attention.self.key.weight ----- False
Parameter: 8bert.encoder.layer.0.attention.self.key.bias ----- False
Parameter: 9bert.encoder.layer.0.attention.self.value.weight ----- False
Parameter: 10bert.encoder.layer.0.attention.self.value.bias ----- False
Parameter: 11bert.encoder.layer.0.attention.output.dense.weight ----- False
Parameter: 12bert.encoder.layer.0.attention.output.dense.bias ----- False
Parameter: 13bert.encoder.layer.0.attention.output.LayerNorm.weight ----- False
Parameter: 14bert.encoder.laye

In [ ]:
# Cargamos el modelo
model_id = "bert-base-cased"
model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)
tokenizer = AutoTokenizer.from_pretrained(model_id)

# El bloque codificador 10 comienza en el índice 165 y
# por ello, congelamos todo antes de ese bloque
for index, (name, param) in enumerate(model.named_parameters()):
    if index < 165:
        param.requires_grad = False

# "trainer" ejecuta el proceso de entrenamiento
trainer = Trainer(
   model=model,
   args=training_args,
   train_dataset=tokenized_train,
   eval_dataset=tokenized_test,
   tokenizer=tokenizer,
   data_collator=data_collator,
   compute_metrics=compute_metrics,
)
trainer.train()
trainer.evaluate()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-4027395977.py:13: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
500,0.493300
1000,0.413400


{'eval_loss': 0.41391509771347046,
 'eval_f1': 0.810077519379845,
 'eval_runtime': 7.1577,
 'eval_samples_per_second': 148.931,
 'eval_steps_per_second': 18.721,
 'epoch': 1.0}

##Entrenar el modelo BERT con un dataset externo

In [ ]:
from google.colab import drive #Accedemos a drive

drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import pandas as pd #Usamos el conjunto de datos IMBD
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Datasets/IMDB_Dataset.csv')
reviews = df['review']

print(reviews.head())

0    One of the other reviewers has mentioned that ...
1    A wonderful little production. <br /><br />The...
2    I thought this was a wonderful way to spend ti...
3    Basically there's a family where a little boy ...
4    Petter Mattei's "Love in the Time of Money" is...
Name: review, dtype: object


In [ ]:
classes=['negative','positive'] #Asignamos las dos clases de los datos

In [ ]:
df['sentiment']= df['sentiment'].apply(lambda x: 1 if x == 'positive' else 0).values
df.head() #Reempazamos las clases por valores numéricos

,review,sentiment
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. <br /><br />The...,1
2,I thought this was a wonderful way to spend ti...,1
3,Basically there's a family where a little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1


In [ ]:
df.rename(columns={'review':'text','sentiment':'label'},inplace=True) #Renombramos las columnas de nuestro Dataframe

In [ ]:
from sklearn.model_selection import train_test_split
df1, df2= train_test_split(df,test_size=0.2,stratify=df['label'],random_state=42)

In [ ]:
X_train, X_test = train_test_split(df2,test_size=0.2,stratify=df2['label'],random_state=42)

In [ ]:
X_train.to_csv('train.csv', index=False) #Guardamos los archivos .csv con los datos de entrenamiento y prueba
X_test.to_csv('test.csv', index=False)

In [ ]:
from datasets import load_dataset #Cargamos los datos previamente generados

tomatoes = load_dataset('csv', data_files={'train': 'train.csv','test': 'test.csv'})

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

In [ ]:
tomatoes

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})